In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix, 
    f1_score, fbeta_score, precision_score, recall_score,
    matthews_corrcoef, brier_score_loss, RocCurveDisplay, roc_auc_score
)
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder,MaxAbsScaler,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.calibration import CalibrationDisplay

from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance, partial_dependence

Goal: We want to maximize amount of money saved by Big G. A full derate costs around \\$4000 for towing, plus the cost of
repairs. The cost of a false positive prediction is about \\$500 due to having the truck oﬀ the road and serviced unnecessarily. Thus, we'll want to optimize our model for recall.

We want to attempt to predict an idle derate at least **2 hours** before it occurs.

In [2]:
faults = pd.read_csv("../data/J1939Faults.csv")

# Dropping columns that are not relevant
faults_clean = faults.drop(columns=['ecuSoftwareVersion', 'ecuSerialNumber', 'ecuModel', 'ecuMake', 'ecuSource', 'MCTNumber', 'ESS_Id', 'actionDescription', 'faultValue'])

faults_clean.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_63237/4289917023.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000


Looking at the first record, here is a breakdown of the important values.

* ESS_Id, actionDescription, ecuSoftwareVersion, ecuSerialNumber, ecuModel, ecuMake, ecuSource, faultValue, and MCTNumber are unlikely to provide any predictive value.
* We can see the time of the event in the **EventTimeStamp** column. Note that this may be different from the **LocationTimeStamp** value, which indicates when the Latitude/Longitude values were recorded.
* The **spn** and **fmi** columns together indicate the type of fault, and there may be a description of that fault in the **eventDescription** column, although this column is sometimes missing.
* Faults are recorded when the light goes on and when it goes off, which is indicated by the **active** column, with True indicating the light turning on and False indicating turning off. The number of times the code has been set or unset is in the **faultValue** column, although this value can be unreliable. 
* Each truck has an identifier, the **EquipmentID** value.
* Each record can be linked to the on-board diagnostics data through the **RecordID** column.

Filtering out vehicles that are likely being serviced: (We'll come back to this)

Service stations are at (36.0666667, -86.4347222), (35.5883333, -86.4438888), and (36.1950, -83.174722)

In [3]:
# Creating a dataframe with service station locations
stations = pd.DataFrame({'lat':[36.0666667, 35.5883333, 36.1950], 'lon':[-86.4347222, -86.4438888, -83.174722]})

In [4]:
# From this article: https://kanoki.org/2019/12/27/how-to-calculate-distance-in-python-and-pandas-using-scipy-spatial-and-distance-functions/
# Vectorized Haversine formula
def haversine_vectorize(lon1, lat1, lon2, lat2):

    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    newlon = lon2 - lon1
    newlat = lat2 - lat1

    haver_formula = np.sin(newlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(newlon/2.0)**2

    dist = 2 * np.arcsin(np.sqrt(haver_formula ))
    mi = 3958 * dist 
    return mi

faults_clean['dist_from_stat_1'] = haversine_vectorize(stations.loc[0]['lon'], stations.loc[0]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_2'] = haversine_vectorize(stations.loc[1]['lon'], stations.loc[1]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_3'] = haversine_vectorize(stations.loc[2]['lon'], stations.loc[2]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])


In [5]:
faults_clean.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp,dist_from_stat_1,dist_from_stat_2,dist_from_stat_3
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000,216.779342,246.957471,200.394786
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000,216.779342,246.957471,200.394786
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000,376.784933,409.225406,437.407343
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000,376.769223,409.209647,437.394356
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000,231.732035,255.969764,376.935409


In [6]:
faults_clean.shape

(1187335, 14)

In [7]:
# Only keep rows that are more than 1 mile away from all 3 of the service stations
faults_clean = faults_clean[(faults_clean['dist_from_stat_1'] > 1) & (faults_clean['dist_from_stat_2'] > 1) & (faults_clean['dist_from_stat_3'] > 1)]


In [8]:
# Drop intermediate columns that we don't need
faults_clean = faults_clean.drop(columns=['dist_from_stat_1','dist_from_stat_2','dist_from_stat_3'])

In [9]:
faults_clean.shape

(1055071, 11)

In [10]:
(1187335-1055071)/1187335

0.11139568866410912

Looks like about 11% of our dataset was collected within 1 mile of a service station.

In [11]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")

diagnostics.head()

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1


Pivoting diagnostic onboard data by fault ID so that it's easier to merge. 

In [12]:
diagnostics[diagnostics['FaultId']==2]

,Id,Name,Value,FaultId
21,22,IgnStatus,True,2
22,23,LampStatus,1279,2


In [13]:
diagnostics_pivoted = diagnostics.pivot(index='FaultId', columns='Name', values='Value')

In [14]:
diagnostics_pivoted.head()

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
1,0,14.21,False,66.48672,423178.7,100.4,11,0,96.74375,0,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


To merge diagnostic info with fault code info, we'll match up the **RecordID** and **FaultId**.

In [15]:
faults_full = pd.merge(faults_clean, diagnostics_pivoted, left_on='RecordID', right_on='FaultId', how='outer')

faults_full.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111.0,17.0,True,2.0,1439,38.857638,-84.626851,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
1,2.0,2015-02-21 11:34:34.000,NaN,629.0,12.0,True,127.0,1439,38.857638,-84.626851,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,False,127.0,1369,41.421250,-87.767361,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,True,127.0,1369,41.421018,-87.767361,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,2015-02-21 11:39:41.000,NaN,4364.0,17.0,False,2.0,1674,38.416481,-89.442638,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


Creating target column:

Our plan will be to make 3 different target columns: 
* Derate in 6 hours
* Derate in 12 hours
* Derate in 24 hours

Then, we'll fit our model on each of the 3 target columns and see which model is most predictive.

The idea is to give the vehicle operator enough warning so that they have enough time to drive to a service station. 0-2 hours of warning probably isn't enough time, so we'll discard those rows when testing our model. As we increase the window larger and larger, we'll probably lose some predictive power. So we'll try to find the sweet spot by evaluating several different time windows.

In [16]:
# Creating a new column to mark whether the fault represents a full derate
faults_full['derate_flag'] = ((faults_full["spn"] == 5246) & (faults_full["active"] == True)).astype('int')

In [17]:
# Converting EventTimeStamp to datetime 
faults_full['EventTimeStamp']= pd.to_datetime(faults_full['EventTimeStamp'])

In [18]:
# Running into an issue with duplicated EquipmentID/EventTimeStamp. 
# The .rolling method can't handle the duplicated EquipmentID and EventTimeStamp columns.
# We'll need to de-duplicate before we apply .rolling
# By taking the max of 'derate_flag', we're ensuring that for each EquipmentID/EventTimeStamp combination, 
#       we're capturing whether at least one derate occurred for that vehicle at that time (if multiple faults occurred simultaneously)
faults_dedup = faults_full.groupby(['EquipmentID', 'EventTimeStamp']).max('derate_flag').reset_index()

In [19]:
faults_dedup.head()

,EquipmentID,EventTimeStamp,RecordID,spn,fmi,activeTransitionCount,Latitude,Longitude,derate_flag
0,301,2015-05-11 13:11:20,49415.0,639.0,2.0,127.0,36.189398,-82.795601,0
1,301,2015-05-13 08:22:32,51363.0,596.0,31.0,3.0,35.872500,-84.475648,0
2,301,2015-05-18 09:34:05,57330.0,3226.0,10.0,6.0,35.972870,-83.920555,0
3,301,2015-05-21 13:57:35,61706.0,639.0,2.0,127.0,36.384953,-86.478379,0
4,301,2015-05-21 14:54:32,61801.0,639.0,2.0,127.0,36.384814,-86.478379,0


In [20]:
# Ensure faults data is sorted by EventTimeStamp
# By sorting the EventTimeStamp in descending order, we're ensuring that we are rolling forwards in time (since .rolling looks at preceding rows)
faults_dedup = faults_dedup.sort_values(by='EventTimeStamp', ascending=False)

# Then we're applying the .rolling method with a certain time window and taking the rolling count of derates.
# The new columns are indicating the total number of derates that happened in the next certain time window for the given vehicle.
target = faults_dedup[['EquipmentID','EventTimeStamp']].copy()
                       
target['total_derates_2_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='2h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_6_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='6h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_12_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='12h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_24_hr'] = faults_dedup.groupby('EquipmentID').rolling(window='24h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)

# If there were 1 or more derates in the given time range, we mark it with "1"; otherwise, "0"
target['derate_2_hr'] = np.where(target['total_derates_2_hr'] > 0, 1, 0)
target['derate_6_hr'] = np.where(target['total_derates_6_hr'] > 0, 1, 0)
target['derate_12_hr'] = np.where(target['total_derates_12_hr'] > 0, 1, 0)
target['derate_24_hr'] = np.where(target['total_derates_24_hr'] > 0, 1, 0)

In [21]:
# Drop intermediate columns that we don't need
target = target.drop(columns=['total_derates_2_hr','total_derates_6_hr','total_derates_12_hr','total_derates_24_hr'])

# Dropping full derates, since we only want to make predictions on events that might lead to a derate, rather than derates themselves
faults_full = faults_full[faults_full['derate_flag'] == 0]

In [22]:
# Next, we'll merge the new target columns into the original DataFrame, merging on EventTimeStamp and EquipmentID
# Any duplicate EventTimeStamp/EquipmentID combinations will have the same value in the target column
faults_full = pd.merge(target, faults_full, on=['EventTimeStamp','EquipmentID'], how='inner')

# Ensure faults_full is sorted in order of EventTimeStamp
faults_full = faults_full.sort_values(by='EventTimeStamp')


In [23]:
# Dropping columns we won't need for our model
faults_full = faults_full.drop(columns=['Latitude','Longitude','RecordID','derate_flag','LocationTimeStamp','eventDescription'])

Exploring missing data:

In [24]:
# Looking at nan proportions overall 
faults_full.isnull().mean().sort_values(ascending=False)

ServiceDistance              0.999813
SwitchedBatteryVoltage       0.899585
FuelTemperature              0.743655
ParkingBrake                 0.665207
Throttle                     0.644931
FuelLevel                    0.569447
AcceleratorPedal             0.545496
CruiseControlActive          0.507487
CruiseControlSetSpeed        0.506494
EngineTimeLtd                0.501613
TurboBoostPressure           0.500011
Speed                        0.499564
EngineOilTemperature         0.499476
FuelRate                     0.498740
FuelLtd                      0.498554
EngineLoad                   0.498487
DistanceLtd                  0.498158
EngineCoolantTemperature     0.498049
BarometricPressure           0.498038
IntakeManifoldTemperature    0.497950
EngineOilPressure            0.497938
EngineRpm                    0.497638
IgnStatus                    0.481304
EventTimeStamp               0.000000
activeTransitionCount        0.000000
active                       0.000000
fmi         

In [25]:
# Dropping service distance since it is mostly nulls
faults_full = faults_full.drop(columns=['ServiceDistance'])

In [26]:
# Investigating values that contain commas
comma_cols = [col for col in faults_full.columns if faults_full[col].astype(str).str.contains(',').any()]
for col in comma_cols:
    print(faults_full[faults_full[col].str.contains(',', na=False)][col].head())

976991     4,8
965121    51,2
958426    99,2
947681    99,6
936402    16,8
Name: AcceleratorPedal, dtype: object
1038224     14,355
1002392     14,355
976991      14,355
975506     14,4275
965121     14,2825
Name: BarometricPressure, dtype: object
1038224    113017,9
1002392    127046,6
976991     136467,8
975506     137217,6
965121     140927,4
Name: DistanceLtd, dtype: object
1038224    186,8
1002392    186,8
975506     183,2
965121     183,2
958426     183,2
Name: EngineCoolantTemperature, dtype: object
1038224    75,98
1002392    73,66
976991     80,62
975506     55,68
965121      66,7
Name: EngineOilPressure, dtype: object
1038224     177,575
1002392    180,1062
976991     189,6687
975506     197,2063
965121     187,3062
Name: EngineOilTemperature, dtype: object
1038224    1570,75
1002392     1725,5
965121     1391,75
955071     1623,25
936402      1414,5
Name: EngineRpm, dtype: object
1038224    2435,65
1002392    2744,05
976991      2956,9
975506      2973,9
965121     3056,35
N

Examining the output above, it appears that all the commas are meant to be decimal points (rather than legitimate commas separating hundreds and thousands, for example). 

In [27]:
# Replacing commas with decimal points
for col in comma_cols:
    faults_full[col] = faults_full[col].str.replace(',', '.', regex=False)

In [28]:
# Split EventTimeStamp column into year, month, day, and hour, then drop EventTimeStamp
faults_full['year'] = faults_full['EventTimeStamp'].dt.year
faults_full['month'] = faults_full['EventTimeStamp'].dt.month
faults_full['weekday'] = faults_full['EventTimeStamp'].dt.weekday
faults_full['hour'] = faults_full['EventTimeStamp'].dt.hour

# Creating a new delta column capturing the time since last fault.
faults_full['min_since_last_fault'] = faults_full.groupby('EquipmentID')['EventTimeStamp'].diff().dt.total_seconds() / 60

In [29]:
# Convert 'object' columns into numerical and string datatypes, as appropriate
numeric_features = ['min_since_last_fault','year','DistanceLtd','EngineTimeLtd','FuelLtd','activeTransitionCount','TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','EngineCoolantTemperature','EngineLoad','EngineOilPressure','EngineOilTemperature','EngineRpm','FuelLevel','FuelRate','FuelTemperature','IntakeManifoldTemperature','Speed','SwitchedBatteryVoltage','Throttle']
for feature in numeric_features:
    faults_full[feature] = faults_full[feature].astype('float32')

binary_features = ['active','CruiseControlActive','IgnStatus','ParkingBrake']
categorical_features = ['LampStatus','spn','fmi']
for feature in categorical_features:
    faults_full[feature] = faults_full[feature].astype('str')

In [30]:
# We'll apply a forward fill to deal with null values in any columns that represent a running count
ffill_features = ['DistanceLtd', 'EngineTimeLtd', 'FuelLtd']
for feature in ffill_features:
    faults_full[feature] = faults_full.groupby('EquipmentID')[feature].ffill().bfill()
    

In [31]:
# We'll use a random fill to deal with nulls in the min_since_last_fault column
faults_full['min_since_last_fault'] = faults_full.groupby('EquipmentID')['min_since_last_fault'].bfill().ffill()

# Get random samples equal to the number of missing values
random_samples = faults_full['min_since_last_fault'].dropna().sample(faults_full['min_since_last_fault'].isnull().sum(), random_state=0)

# Match indices of random samples with NaN indices
random_samples.index = faults_full[faults_full['min_since_last_fault'].isnull()].index

# Fill missing values with the randomly generated values
faults_full.loc[faults_full['min_since_last_fault'].isnull(), 'min_since_last_fault'] = random_samples

In [32]:
# For components like hours (0–23), we'll use sine and cosine transformations can help the model understand that hour 23 is close to hour 0
# Article here: https://feature-engine.trainindata.com/en/1.8.x/user_guide/creation/CyclicalFeatures.html

# Define transformation functions
def sin_transformer(period):
    return FunctionTransformer(lambda x: np.sin(x / period * 2 * np.pi), feature_names_out="one-to-one")

def cos_transformer(period):
    return FunctionTransformer(lambda x: np.cos(x / period * 2 * np.pi), feature_names_out="one-to-one")

# Numeric pipeline
numeric_transformer = Pipeline(steps=[
    # Applying a simple imputer to fill missing values
    ("imputer", IterativeImputer(random_state=0, verbose=2))),
    # Applying a MaxAbsScaler (better than StandardScaler for sparse inputs or inputs that are not normally distributed)
    ("scaler", MaxAbsScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Cyclical pipeline for features that repeat in a cycle (hours, days, and months)
cyclical_transformer = ColumnTransformer(transformers=[
    ("month_sin", sin_transformer(12), ["month"]),
    ("month_cos", cos_transformer(12), ["month"]),
    ("weekday_sin", sin_transformer(7), ["weekday"]),
    ("weekday_cos", cos_transformer(7), ["weekday"]),
    ("hour_sin", sin_transformer(24), ["hour"]),
    ("hour_cos", cos_transformer(24), ["hour"]),
])

# Combine all the transformations into 1 preprocessor
# Using ColumnTransformer to select specific columns and apply transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("cycle", cyclical_transformer, ["month", "weekday", "hour"])
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

pipe = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

In [33]:
# Anything before 1/1/2019 is our training/validation data
train_set = faults_full[~(faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date())]

# Converting EventTimeStamp to datetime
train_set['EventTimeStamp'] = pd.to_datetime(train_set['EventTimeStamp'])

# Anything after 1/1/2019 is the test data
test_set = faults_full[faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date()]

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_63237/2369204415.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_set['EventTimeStamp'] = pd.to_datetime(train_set['EventTimeStamp'])


Here, we'll split our data using 5-fold validation and apply the pre-processor pipeline (scaling and filling in missing values).

In [35]:
# Create 5 equal-sized arrays from train_set
parts = np.array_split(train_set, 5)

# Initialize an empty list to keep track of the training data
train_parts = []

for i in range(4):
    # The training set is every part up to the current index combined
    train_parts = [parts[j] for j in range(4) if j <= i]
    train_set = pd.concat(train_parts)
    
    # The validation set is always the part corresponding to the next index
    val_set = parts[i + 1]

    # Fit and transform the train set
    train_clean = pipe.fit_transform(train_set)
    
    # Transform the validation set 
    val_clean = pipe.transform(val_set)

    # Attach feature names
    feature_names = pipe.get_feature_names_out()
    train_clean_df = pd.DataFrame(train_clean, columns=feature_names)
    val_clean_df = pd.DataFrame(val_clean, columns=feature_names)

    # Remove events that are within 2 hours of a derate from the validation set 
    val_clean_df = val_clean_df[val_clean_df['derate_2_hr'] == 0]

    # Add sin and cos cyclical features together 
    for time in ('month','weekday','hour'):
        col_name = f'{time}_cyclical'
        train_clean_df = train_clean_df.assign(
            **{col_name: train_clean_df[f'{time}_sin__{time}'] + train_clean_df[f'{time}_cos__{time}']}
        ).drop(columns=[f'{time}_sin__{time}', f'{time}_cos__{time}'])
        val_clean_df = val_clean_df.assign(
            **{col_name: val_clean_df[f'{time}_sin__{time}'] + val_clean_df[f'{time}_cos__{time}']}
        ).drop(columns=[f'{time}_sin__{time}', f'{time}_cos__{time}'])
        
    # Save the cleaned data 
    train_clean_df.to_pickle(f'../data/train_set_{i+1}.pkl')
    val_clean_df.to_pickle(f'../data/val_set_{i+1}.pkl')

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [34]:
def calculate_cost_savings(evaluation_df):
    """
    Returns the amount of money we are saving Big G by implementing our model
    """

    evaluation_df['date']= pd.to_datetime(evaluation_df['EventTimeStamp']).dt.date
    model_cost_total = evaluation_df.groupby(['EquipmentID', 'date']).agg({'actual_derate': 'max', 'pred_derate': 'max'}).sort_values('actual_derate', ascending=False)
    
    # Create a for loop to calculate cost based on formulas
    total_cost_list = []
    for row in range(len(model_cost_total)):
        eval_row = model_cost_total.iloc[row]
        if (eval_row['actual_derate'] == 0) & (eval_row['pred_derate'] == 1):
            total_cost_list.append(-500)
        elif (eval_row['actual_derate'] == 1) & (eval_row['pred_derate'] == 1):
            total_cost_list.append(4000)
        else:
            total_cost_list.append(0)
        
    return np.sum(total_cost_list)

Now that we have our pre-processed cross-validation datasets, we'll fit a model to each of the sets and evaluate its effectiveness. We'll apply 5-fold cross-validation to evaluate how well our model does. 
Good article here: https://www.geeksforgeeks.org/machine-learning/k-fold-cross-validation-in-machine-learning/

In [35]:
# Initialize a dataframe to store model evaluation metrics 
model_evaluation_results = pd.DataFrame(columns=['fold', 'time_window', 'upweight', 'oversample', 'f-beta', 'ROC-AUC', 'precision', 'recall', 'false positives', 'true positives', 'cost_savings']) 

targets = ['derate_6_hr','derate_12_hr','derate_24_hr'] 

for target in targets:     
    print(f'TARGET: {target}')      
    
    for strategy in ('upweight','oversample'):         
        print(f'BALANCING STRATEGY: {strategy}')         

        for i in range(1, 5):
            # Read in pre-processed train and validation data 
            train_set = pd.read_pickle(f'../data/train_set_{i}.pkl') 
            val_set = pd.read_pickle(f'../data/val_set_{i}.pkl') 
                 
            # Convert columns into numerical and bool datatypes, as appropriate 
            numeric_features = ['min_since_last_fault','year','DistanceLtd','EngineTimeLtd','FuelLtd','activeTransitionCount','TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','EngineCoolantTemperature','EngineLoad','EngineOilPressure','EngineOilTemperature','EngineRpm','FuelLevel','FuelRate','FuelTemperature','IntakeManifoldTemperature','Speed','SwitchedBatteryVoltage','Throttle','derate_6_hr','derate_12_hr','derate_24_hr'] 
            for feature in numeric_features: 
                train_set[feature] = train_set[feature].astype('float32') 
                val_set[feature] = val_set[feature].astype('float32') 
                         
            bools = [col for col in train_set.columns if col not in numeric_features and col not in ('EquipmentID','EventTimeStamp') and not col.startswith('derate')] 
            train_set[bools] = train_set[bools].astype(bool) 
            val_set[bools] = val_set[bools].astype(bool) 
                     
            # Split into x (features) and y (target) 
            X_train = train_set[[col for col in train_set.columns if not col.startswith('derate') and col not in ('EquipmentID','EventTimeStamp')]] 
            y_train = train_set[target] 
                     
            X_val = val_set[[col for col in val_set.columns if not col.startswith('derate') and col not in ('EquipmentID','EventTimeStamp')]] 
            y_val = val_set[target] 
                 
            if strategy == 'upweight': 
                upweight = 1 
                oversample = 0 
                # Defining the scale_pos_weight hyperparameter 
                if (train_set[target].value_counts().loc[1] == 0): 
                    param = train_set[target].value_counts().loc[0] 
                else: 
                    param = train_set[target].value_counts().loc[0] / train_set[target].value_counts().loc[1] 
                                
                # Initialize and fit a model 
                model = XGBClassifier(scale_pos_weight=param).fit(X_train, y_train) 
            else: 
                upweight = 0 
                oversample = 1 
                # Apply SMOTE oversampling to deal with imbalanced classes 
                smote = SMOTE(random_state=321) 
                X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train) 
                                
                # Initialize and fit a model 
                model = XGBClassifier().fit(X_train_smote, y_train_smote) 
                         
            # Generate predictions on the validation set 
            y_pred = model.predict(X_val) 

            # Generate predicted probabiliites
            y_pred_proba = model.predict_proba(X_val)
            
            # Isolate probability of a derate
            y_prob_derate = y_pred_proba[:, 1]
                 
            # Prepare a dataframe to assess cost savings 
            evaluation_df = val_set[['EquipmentID','EventTimeStamp']].copy() 
            evaluation_df['pred_derate'] = y_pred 
            evaluation_df['actual_derate'] = y_val 
            evaluation_df['pred_derate'] = evaluation_df['pred_derate'].astype(int) 
            evaluation_df['actual_derate'] = evaluation_df['actual_derate'].astype(int) 
                     
            # Calculate metrics for this specific fold
            cost_savings = calculate_cost_savings(evaluation_df) 
            prec = precision_score(y_val, y_pred) 
            rec = recall_score(y_val, y_pred) 
            fbeta = fbeta_score(y_val, y_pred, beta=8) 
            roc_auc = roc_auc_score(y_val, y_prob_derate)
            
            cm = confusion_matrix(y_val, y_pred) 
            tn, fp, fn, tp = cm.ravel() 
             
            # Append 1 row per fold to the results dataframe
            model_evaluation_results.loc[len(model_evaluation_results)] = [i, target, upweight, oversample, fbeta, roc_auc, prec, rec, fp, tp, cost_savings]

TARGET: derate_6_hr
BALANCING STRATEGY: upweight
BALANCING STRATEGY: oversample
TARGET: derate_12_hr
BALANCING STRATEGY: upweight
BALANCING STRATEGY: oversample
TARGET: derate_24_hr
BALANCING STRATEGY: upweight
BALANCING STRATEGY: oversample


In [36]:
model_evaluation_results

,fold,time_window,upweight,oversample,f-beta,ROC-AUC,precision,recall,false positives,true positives,cost_savings
0,1,derate_6_hr,1,0,0.000000,0.408778,0.000000,0.000000,402,0,-71000
1,2,derate_6_hr,1,0,0.000000,0.553078,0.000000,0.000000,5648,0,-669500
2,3,derate_6_hr,1,0,0.042349,0.674255,0.002150,0.059829,3249,7,-639000
3,4,derate_6_hr,1,0,0.005366,0.637131,0.000146,0.012195,6865,1,-1405500
4,1,derate_6_hr,0,1,0.023669,0.468328,0.000915,0.038710,6551,6,-686500
5,2,derate_6_hr,0,1,0.009598,0.429427,0.000225,0.027397,8871,2,-1455000
6,3,derate_6_hr,0,1,0.083303,0.654408,0.004075,0.119658,3422,14,-678500
7,4,derate_6_hr,0,1,0.031903,0.622410,0.000777,0.085366,9007,7,-2329500
8,1,derate_12_hr,1,0,0.000000,0.480841,0.000000,0.000000,397,0,-116000
9,2,derate_12_hr,1,0,0.000000,0.560780,0.000000,0.000000,9220,0,-1058000


In [37]:
# Average metrics across all folds
summary_df = model_evaluation_results.groupby(
    ['time_window', 'upweight', 'oversample']
).mean().reset_index().drop(columns='fold')

summary_df

,time_window,upweight,oversample,f-beta,ROC-AUC,precision,recall,false positives,true positives,cost_savings
0,derate_12_hr,0,1,0.030506,0.551253,0.001548,0.047592,7358.25,8.25,-1284875.0
1,derate_12_hr,1,0,0.016207,0.584128,0.001124,0.023292,5069.50,4.00,-848750.0
2,derate_24_hr,0,1,0.053763,0.575643,0.003970,0.070342,7532.25,23.25,-1358625.0
3,derate_24_hr,1,0,0.028091,0.581968,0.005348,0.033448,4184.00,10.00,-662625.0
4,derate_6_hr,0,1,0.037118,0.543643,0.001498,0.067783,6962.75,7.25,-1287375.0
5,derate_6_hr,1,0,0.011929,0.568311,0.000574,0.018006,4041.00,2.00,-696250.0


Seeing how well the model does on our "real-world" data:

In [ ]:
# Redefine train set since we've already transformed it during k-fold
train_set = faults_full[~(faults_full['EventTimeStamp'].dt.date >= pd.to_datetime('2019-01-01').date())]

# Converting EventTimeStamp to datetime
train_set['EventTimeStamp'] = pd.to_datetime(train_set['EventTimeStamp'])

# Initialize a dataframe to store model evaluation metrics
model_evaluation_results = pd.DataFrame(columns=['time_window', 'upweight', 'oversample', 'f-beta', 'ROC-AUC', 'precision', 'recall', 'false positives', 'true positives', 'cost_savings'])

# We'll use the 24 hour time window for our target since it performed the best during k-fold validation
target = 'derate_24_hr'

# Remove events that are within 2 hours of a derate for the test set 
test_set = test_set[test_set['derate_2_hr'] == 0]

# Fit and transform the train set using our preprocessing pipeline
train_clean = pipe.fit_transform(train_set)

# Transform the test set
test_clean = pipe.transform(test_set)

# Get feature names from preprocessing pipeline
feature_names = pipe.get_feature_names_out()

# Attach feature names
train_clean_df = pd.DataFrame(train_clean)
train_clean_df.columns = feature_names

test_clean_df = pd.DataFrame(test_clean)
test_clean_df.columns = feature_names

# Convert columns into numerical and bool datatypes, as appropriate 
numeric_features = ['min_since_last_fault','year','DistanceLtd','EngineTimeLtd','FuelLtd','activeTransitionCount','TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','EngineCoolantTemperature','EngineLoad','EngineOilPressure','EngineOilTemperature','EngineRpm','FuelLevel','FuelRate','FuelTemperature','IntakeManifoldTemperature','Speed','SwitchedBatteryVoltage','Throttle','derate_6_hr','derate_12_hr','derate_24_hr'] 
for feature in numeric_features: 
    train_clean_df[feature] = train_clean_df[feature].astype('float32') 
    test_clean_df[feature] = test_clean_df[feature].astype('float32') 
             
bools = [col for col in train_clean_df.columns if col not in numeric_features and col not in ('EquipmentID','EventTimeStamp') and not col.startswith('derate')] 
train_clean_df[bools] = train_clean_df[bools].astype(bool) 
test_clean_df[bools] = test_clean_df[bools].astype(bool) 

# Split into x (features) and y (target)
X_train = train_clean_df[[col for col in train_clean_df.columns if not col.startswith('derate') and col not in ('EquipmentID','EventTimeStamp')]]
y_train = train_clean_df[target]

X_test = test_clean_df[[col for col in test_clean_df.columns if not col.startswith('derate') and col not in ('EquipmentID','EventTimeStamp')]]
y_test = test_clean_df[target]

# We'll use the upweighting approach since it performed better during k-fold validation
param = train_set[target].value_counts().loc[0]/train_set[target].value_counts().loc[1]

# Initialize and fit a model
model = XGBClassifier(scale_pos_weight=param).fit(X_train, y_train)

# Generate predictions on the test set using the model
y_pred = model.predict(X_test)

# Generate predicted probabiliites
y_pred_proba = model.predict_proba(X_val)

# Isolate probability of a derate
y_prob_derate = y_pred_proba[:, 1]

# Prepare a dataframe to assess cost savings
evaluation_df = test_set[['EquipmentID','EventTimeStamp']].copy()
evaluation_df['pred_derate']= y_pred
evaluation_df['actual_derate']= y_test

# Calculate true positives and false positives 
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Append results to model_evaluation_results
model_evaluation_results.loc[len(model_evaluation_results)] = [target, 1, 0, fbeta_score(y_test, y_pred, beta=8), roc_auc_score(y_val, y_prob_derate), precision_score(y_test, y_pred), recall_score(y_test, y_pred), fp, tp, calculate_cost_savings(evaluation_df)]

# Print model evaluation results
model_evaluation_results

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_63237/1601375010.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_set['EventTimeStamp'] = pd.to_datetime(train_set['EventTimeStamp'])


Model Interpretability/Explainability:

In [ ]:
# Printing top features by permutation importance
pd.DataFrame({
    'variable': X_test.columns,
    'importance': permutation_importance(model, X_test.sample(n=1000, random_state=0), y_test.sample(n=1000, random_state=0), random_state = 0, scoring='f1')['importances_mean']
}).sort_values('importance', ascending = False)

In [ ]:
# Plotting features by average shap score
explainer = shap.TreeExplainer(xgb)
explanation = explainer(X_test)
shap.plots.bar(explanation)